In [ ]:
import pandas as pd
import numpy as np

from dataclasses import dataclass, field
from typing import Optional, Protocol
from ortools.sat.python import cp_model

## System structure

```mermaid
flowchart TD;

st([ST]) <-.-> coll[[Collector]]

coll <-->  inv1[Inverter #1]
coll <-->  inv2[Inverter #2]
coll <-->  inv3[Inverter #3]

inv1 --- pv1["PV #1"]
inv1 --- cons1["Consumer #1"]

inv2 --- pv2["PV #2"]
inv2 --> cons2["Consumer #2"]
inv2 <--> bess1["BESS #1"]

inv3 <--> bess2["BESS #2"]
inv3 <--> bess3["BESS #3"]
```

*Inverters* have internal power limit (connections between devices), as well as output limit (connection to collector);

*Collector* is modelled as inverter with "unlimited" internal power capacity and output is limited by ST connection;

*PVs* output power according to defined profile (in this example based on Global Horizontal Irradiance from typical meteorological year);

*Consumers* consume power according to defined profile (in this example based on average household consumption);

*BESSes* operate under number of constraints, such as energy capacity, max charge/discharge power, cycles limit and efficiency.
  - *BESS #1* has mediocre parameters;
  - *BESS #2* is almost fully charged high performance battery (e.g. Tesla), change level at time horizon is not defined;
  - *BESS #3* is almost empty high capacity battery (e.g. Tesla), expected to be fully charged at time horizon;

The example is meant to demonstrate energy transfer between BESSes with power exceeding inverter's output power, but within its internal limits. Same goes for collector level energy transfer between inverters and ST.

## Inputs

In [ ]:
df = pd.read_excel("input_data.xlsx", index_col=0)

df["PV#1"] = (df["ghi"] * 16).astype(int)
df["CONS#1"] = (df["avg"] / 100 * 10_000).astype(int)

df["PV#2"] = (df["ghi"] * 24).astype(int)
df["CONS#2"] = (df["avg"] / 100 * 7_000).astype(int)

df

## Classes

In [ ]:
from ortools.sat.python.cp_model import CpModel, CpSolver


class Device(Protocol):
    v_power: list[cp_model.IntVar | int]
    """ units: watt-hours"""

    def add_to_model(self, time_horizon: int, model: cp_model.CpModel): ...
    def evaluate(self, solver: cp_model.CpSolver): ...


@dataclass
class FixedDevice(Device):
    name: str
    power_profile: list[int]

    def add_to_model(self, time_horizon: int, model: cp_model.CpModel):
        self.v_power = [self.power_profile[ts] for ts in range(time_horizon)]

    def evaluate(self, solver: cp_model.CpSolver):
        self.power = np.array(self.v_power)


@dataclass
class Battery(Device):
    name: str
    capacity: int
    max_charge_power: int
    max_discharge_power: int
    start_energy: int = 0
    end_energy: Optional[int] = None
    cycles_limit: Optional[float] = 2.0
    with_efficiency: bool = True
    charge_efficiency: float = 0.90
    discharge_efficiency: float = 0.90

    def __post_init__(self):
        assert self.max_charge_power > 0
        assert self.max_discharge_power < 0

        if self.with_efficiency:
            self.multiplier = 100
            self.in_coef = int(self.charge_efficiency * self.multiplier)
            self.out_coef = int(1 / self.discharge_efficiency * self.multiplier)
        else:
            self.multiplier = 1
            self.in_coef = 1
            self.out_coef = 1

    def add_to_model(self, time_horizon: int, model: CpModel):
        self.v_energy = []
        self.v_power = []

        v_in_power = []
        v_out_power = []

        for ts in range(time_horizon):
            self.v_energy.append(
                model.new_int_var(0, self.capacity, f"{self.name}_level_{ts:03d}")
            )
            v_in_power.append(
                model.new_int_var(
                    0, self.max_charge_power, f"{self.name}_in_power_{ts:03d}"
                )
            )
            v_out_power.append(
                model.new_int_var(
                    self.max_discharge_power, 0, f"{self.name}_out_power_{ts:03d}"
                )
            )
            self.v_power.append(
                model.new_int_var(
                    self.max_discharge_power,
                    self.max_charge_power,
                    f"{self.name}_power_{ts:03d}",
                )
            )

            model.add(self.v_power[ts] == v_in_power[ts] + v_out_power[ts])

            # times 4 due to 15 min intervals
            model.add(
                4 * self.start_energy * self.multiplier
                + sum(v_in_power[:ts]) * self.in_coef
                + sum(v_out_power[:ts]) * self.out_coef
                == 4 * self.v_energy[ts] * self.multiplier
            )

            if self.cycles_limit:
                model.add(
                    sum(v_in_power[ts - 24 * 4 : ts + 1])
                    <= int(4 * self.capacity * self.cycles_limit)
                )
                model.add(
                    sum(v_out_power[ts - 24 * 4 : ts + 1])
                    >= -int(4 * self.capacity * self.cycles_limit)
                )

        if self.end_energy:
            model.add(self.v_energy[-1] >= self.end_energy)

    def evaluate(self, solver: cp_model.CpSolver):
        self.power = np.array([solver.value(p) for p in self.v_power])
        self.energy = np.array([solver.value(l) for l in self.v_energy])


@dataclass
class Inverter(Device):
    name: str
    max_output_power: int
    max_internal_power: int
    devices: list[Device]

    def add_to_model(self, time_horizon: int, model: cp_model.CpModel):
        for dev in self.devices:
            dev.add_to_model(time_horizon, model)

        self.v_power = []
        for ts in range(time_horizon):
            p = model.new_int_var(
                -self.max_output_power,
                self.max_output_power,
                f"{self.name}_out_{ts:03d}",
            )
            self.v_power.append(p)

            p_dev = [dev.v_power[ts] for dev in self.devices]
            model.add(p == sum(p_dev))

            p_internal_max = model.new_int_var(
                0, self.max_internal_power, f"{self.name}_internal_max_{ts:03d}"
            )
            model.add_max_equality(p_internal_max, p_dev)

            p_internal_min = model.new_int_var(
                -self.max_internal_power, 0, f"{self.name}_internal_min_{ts:03d}"
            )
            model.add_min_equality(p_internal_min, p_dev)

    def evaluate(self, solver: CpSolver):
        for dev in self.devices:
            dev.evaluate(solver)

        self.power = np.array([solver.value(p) for p in self.v_power])

## Devices

In [ ]:
pv1 = FixedDevice(
    name="PV #1",
    power_profile=list(-df["PV#1"]),
)
cons1 = FixedDevice(
    name="Consumer #1",
    power_profile=list(df["CONS#1"]),
)
inv1 = Inverter(
    name="Inv #1",
    max_output_power=5_000,
    max_internal_power=10_000,
    devices=[pv1, cons1],
)

pv2 = FixedDevice(
    name="PV #2",
    power_profile=list(-df["PV#2"]),
)
cons2 = FixedDevice(
    name="Consumer #2",
    power_profile=list(df["CONS#2"]),
)
bess1 = Battery(
    name="BESS #1",
    capacity=20_000,
    max_charge_power=3_500,
    max_discharge_power=-5_000,
    start_energy=10_000,
    end_energy=10_000,
    cycles_limit=2.0,
    with_efficiency=True,
)
inv2 = Inverter(
    name="Inv #2",
    max_output_power=6_000,
    max_internal_power=12_000,
    devices=[pv2, cons2, bess1],
)

bess2 = Battery(
    name="BESS #2",
    capacity=50_000,
    max_charge_power=12_000,
    max_discharge_power=-12_000,
    start_energy=40_000,
    end_energy=None,
    cycles_limit=2.0,
    with_efficiency=True,
)
bess3 = Battery(
    name="BESS #3",
    capacity=50_000,
    max_charge_power=12_000,
    max_discharge_power=-12_000,
    start_energy=3_000,
    end_energy=45_000,
    cycles_limit=2.0,
    with_efficiency=True,
)
inv3 = Inverter(
    name="Inv #3",
    max_output_power=6_000,
    max_internal_power=12_000,
    devices=[bess2, bess3],
)

coll = Inverter(
    name="Collector",
    max_internal_power=100_000,  # "unlimited" internal power
    max_output_power=5_200,  # connection to ST
    devices=[inv1, inv2, inv3],
)

## Solving the model

In [ ]:
# prices in "EUR per Watt_15min" (in contrast to EUR per kiloWatt_Hour)
buy_prices = list(df["buy"] / 1000 / 4)
sell_prices = list(df["sell"] / 1000 / 4)
TH = len(buy_prices)

# define CP model
model = cp_model.CpModel()
coll.add_to_model(TH, model) # collector includes whole nested hierarchy

v_costs = []
for ts in range(TH):
    v_buy = model.new_int_var(0, coll.max_output_power, f"buy_power_{ts:03d}")
    model.add_max_equality(v_buy, [coll.v_power[ts], 0])

    v_sell = model.new_int_var(-coll.max_output_power, 0, f"sell_power_{ts:03d}")
    model.add_min_equality(v_sell, [coll.v_power[ts], 0])

    v_costs.append(v_buy * buy_prices[ts] + v_sell * sell_prices[ts])

model.minimize(sum(v_costs))

# solve CP model
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30
res = solver.solve(model, solution_callback=cp_model.ObjectiveSolutionPrinter())

print(f"Status: {solver.StatusName(res)}")
if res == cp_model.OPTIMAL or res == cp_model.FEASIBLE:
    print(f"Objective value: {solver.ObjectiveValue():.2f} EUR")

# evaluate variables
coll.evaluate(solver) # includes nested hierarchy
costs = np.array([solver.float_value(c) for c in v_costs])

## Plots

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter, HourLocator

fig, axs = plt.subplots(nrows=6, figsize=(9, 9), sharex=True)

axs[0].step(df.index, df["buy"], where="post", label="Buy", color="C3")
axs[0].step(df.index, df["sell"], where="post", label="Sell", color="C2")
axs[0].set_ylabel("Price, $EUR/kWh$")

for i, grp in enumerate(
    [
        [coll, inv1, inv2, inv3],
        [inv1, pv1, cons1],
        [inv2, pv2, cons2, bess1],
        [inv3, bess2, bess3],
    ],
    start=1,
):
    for di, dev in enumerate(grp):
        axs[i].step(
            df.index,
            dev.power / 1000,
            where="post",
            label=dev.name,
            alpha=1.0 if di == 0 else 0.5,
        )
    axs[i].set_ylabel("Power, $kW$")

for dev in [bess1, bess2, bess3]:
    axs[-1].step(df.index, dev.energy / 1000, where="post", label=dev.name)
axs[-1].set_ylabel("Stored energy, $kWh$")

for ax in axs:
    ax.grid()
    ax.set_xlim(
        df.index.min() - pd.Timedelta(minutes=30),
        df.index.max() + pd.Timedelta(minutes=45),
    )
    ax.xaxis.set_tick_params(which="both", labelbottom=True)
    ax.xaxis.set_major_formatter(DateFormatter("%H:%M"))
    ax.xaxis.set_major_locator(HourLocator(range(0, 24, 3)))
    ax.xaxis.set_minor_locator(HourLocator())
    ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))

## Result

In [ ]:
res = pd.DataFrame(index = df.index)

res["ST -- Coll"] = coll.power

res["Coll -- Inv#1"] = inv1.power
res["Coll -- Inv#2"] = inv2.power
res["Coll -- Inv#3"] = inv3.power

res["Inv#1 -- PV#1"] = pv1.power
res["Inv#1 -- Cons#1"] = cons1.power

res["Inv#2 -- PV#2"] = pv2.power
res["Inv#2 -- Cons#2"] = cons2.power
res["Inv#2 -- BESS#1"] = bess1.power

res["Inv#3 -- BESS#2"] = bess2.power
res["Inv#3 -- BESS#3"] = bess3.power

res.to_excel("output_data.xlsx")
res